In [37]:
#imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow

In [38]:
#data
transactions = pd.read_csv("../data/transactions_2016_2017.csv")
train = pd.read_csv("../data/customer_clv_train.csv")
test = pd.read_csv("../data/customer_clv_test.csv")

/var/folders/3m/cw1m5lkx3ml5zf_qkrwvpxph0000gn/T/ipykernel_69434/2271748159.py:2: DtypeWarning: Columns (0: prod_size) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv("../data/transactions_2016_2017.csv")


In [39]:
transactions["order_date"] = pd.to_datetime(transactions["order_date"])
transactions["pack_date"] = pd.to_datetime(transactions["pack_date"])

# RFM Features

In [40]:
reference_date = transactions["order_date"].max()

In [41]:
# identify returns and purchases
transactions["is_return"] = transactions["sale_revenue"] < 0
transactions["is_purchase"] = transactions["sale_revenue"] > 0

In [42]:
# create customer features
customer_features = (
    transactions
    .groupby("cust_id")
    .agg(
        # Monetary
        total_revenue=("sale_revenue", "sum"),
        avg_item_revenue=("sale_revenue", "mean"),
        max_item_revenue=("sale_revenue", "max"),

        # Frequency
        n_orders=("sale_id", "nunique"),
        n_items=("sale_id", "count"),

        # Discounts
        total_discount=("sale_discount_applied", "sum"),

        # Returns
        n_returns=("is_return", "sum"),

        # Product diversity
        n_products=("prod_id", "nunique"),
        n_brands=("prod_brand", "nunique"),
        n_colors=("prod_color", "nunique"),
        n_categories=("prod_type_3", "nunique"),

        # Size behaviour
        avg_size=("prod_size", lambda x: pd.to_numeric(x, errors="coerce").mean()),
        size_std=("prod_size", lambda x: pd.to_numeric(x, errors="coerce").std()),
        n_sizes=("prod_size", "nunique"),

        # Dates
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max")
    )
    .reset_index()
)

In [43]:
customer_features["recency_days"] = (
    reference_date - customer_features["last_purchase"]
).dt.days

customer_features["customer_age_days"] = (
    customer_features["last_purchase"] - customer_features["first_purchase"]
).dt.days

customer_features = customer_features.drop(
    columns=["first_purchase", "last_purchase"]
)

In [44]:
customer_features["items_per_order"] = (
    customer_features["n_items"] / customer_features["n_orders"]
)

customer_features["return_rate"] = (
    customer_features["n_returns"] / customer_features["n_items"]
)

customer_features["avg_order_value"] = (
    customer_features["total_revenue"] / customer_features["n_orders"]
)

In [45]:
customer_features["size_std"] = customer_features["size_std"].fillna(0)

In [46]:
# intensity features
customer_features["orders_per_day"] = (
    customer_features["n_orders"] /
    (customer_features["customer_age_days"] + 1)
)

customer_features["items_per_day"] = (
    customer_features["n_items"] /
    (customer_features["customer_age_days"] + 1)
)

customer_features["revenue_per_day"] = (
    customer_features["total_revenue"] /
    (customer_features["customer_age_days"] + 1)
)

customer_features["frequency_recency_ratio"] = (
    customer_features["n_orders"] /
    (customer_features["recency_days"] + 1)
)

customer_features["revenue_recency_ratio"] = (
    customer_features["total_revenue"] /
    (customer_features["recency_days"] + 1)
)

customer_features["discount_per_item"] = (
    customer_features["total_discount"] /
    (customer_features["n_items"] + 1)
)

customer_features["discount_per_order"] = (
    customer_features["total_discount"] /
    (customer_features["n_orders"] + 1)
)

In [47]:
# Average Gaps

transactions_sorted = transactions.sort_values(["cust_id", "order_date"]).copy()

transactions_sorted["prev_order_date"] = (
    transactions_sorted.groupby("cust_id")["order_date"].shift()
)

transactions_sorted["days_between_orders"] = (
    transactions_sorted["order_date"] - transactions_sorted["prev_order_date"]
).dt.days

avg_gap = transactions_sorted.groupby("cust_id")["days_between_orders"].mean()
max_gap = transactions_sorted.groupby("cust_id")["days_between_orders"].max()

customer_features["avg_days_between_orders"] = customer_features["cust_id"].map(avg_gap)
customer_features["max_days_between_orders"] = customer_features["cust_id"].map(max_gap)

customer_features["avg_days_between_orders"] = (
    customer_features["avg_days_between_orders"].fillna(customer_features["recency_days"])
)
customer_features["max_days_between_orders"] = (
    customer_features["max_days_between_orders"].fillna(customer_features["recency_days"])
)

In [48]:
customer_features["ever_returned"] = (customer_features["n_returns"] > 0).astype(int)

customer_features["returns_per_order"] = (
    customer_features["n_returns"] /
    (customer_features["n_orders"] + 1)
)

# Features added by me

In [49]:
# add delivery_days to transactions for avg_delivery_days feature
transactions["delivery_days"] = (transactions["pack_date"] - transactions["order_date"]).dt.days

# aggregate new features at customer level
new_features = (
    transactions
    .groupby("cust_id")
    .agg(
        web_only_rate=("prod_web_only", "mean"),
        outlet_rate=("prod_outlet", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        n_seasons=("prod_season", "nunique"),
    )
    .reset_index()
)

# merge into customer_features
customer_features = customer_features.merge(new_features, on="cust_id", how="left")


In [50]:
# Spending trend features
# find midpoint of the observation period
midpoint_date = transactions["order_date"].min() + (
    transactions["order_date"].max() - transactions["order_date"].min()
) / 2

# Split transactions into first half and second half based on the midpoint date
transactions['is_second_half'] = transactions['order_date'] >= midpoint_date

# aggregate per customer 
first_half_revenue = (transactions[transactions['is_second_half'] == False].groupby('cust_id')['sale_revenue'].sum())
second_half_revenue = (transactions[transactions['is_second_half'] == True].groupby('cust_id')['sale_revenue'].sum())

# Spending trend: positive if spending increased in the second half, negative if it decreased
customer_features['spending_trend'] = customer_features['cust_id'].map(second_half_revenue - first_half_revenue)

# filling Nans with 0 (customers with no transactions in one half will have NaN, but we can interpret that as no change in spending)
customer_features["spending_trend"] = customer_features["spending_trend"].fillna(0)

In [51]:
# === Additional engineered features ===

# 1. Seasonality: first/last purchase month
fp_lp = transactions.groupby("cust_id")["order_date"].agg(["min", "max"])
customer_features["first_purchase_month"] = customer_features["cust_id"].map(fp_lp["min"].dt.month)
customer_features["last_purchase_month"] = customer_features["cust_id"].map(fp_lp["max"].dt.month)

# 2. Active months: how many distinct calendar months had at least one order
active_months = (
    transactions
    .assign(ym=transactions["order_date"].dt.to_period("M"))
    .groupby("cust_id")["ym"]
    .nunique()
)
customer_features["active_months"] = customer_features["cust_id"].map(active_months)

# 3. Revenue lost due to returns (returned items have negative sale_revenue)
return_rev_lost = (
    transactions[transactions["is_return"]]
    .groupby("cust_id")["sale_revenue"]
    .sum()
    .abs()
)
customer_features["return_revenue_lost"] = customer_features["cust_id"].map(return_rev_lost).fillna(0)

# 4. Average revenue per item, excluding returns
purchase_agg = (
    transactions[transactions["is_purchase"]]
    .groupby("cust_id")["sale_revenue"]
    .agg(["sum", "count"])
)
customer_features["avg_revenue_excl_returns"] = (
    customer_features["cust_id"]
    .map(purchase_agg["sum"] / purchase_agg["count"])
    .fillna(0)
)

# 5. Brand concentration (Herfindahl index) — higher value = more loyal to a single brand
def herfindahl(x):
    p = x.value_counts(normalize=True)
    return (p**2).sum()

brand_conc = transactions.groupby("cust_id")["prod_brand"].apply(herfindahl)
customer_features["brand_concentration"] = customer_features["cust_id"].map(brand_conc)

# 6. Category concentration (Herfindahl over prod_type_3)
cat_conc = transactions.groupby("cust_id")["prod_type_3"].apply(herfindahl)
customer_features["category_concentration"] = customer_features["cust_id"].map(cat_conc)

# 7. Relative discount rate: fraction of gross price that was discounted
customer_features["discount_rate"] = (
    customer_features["total_discount"] /
    (customer_features["total_revenue"] + customer_features["total_discount"] + 1e-6)
)

# 8. New-season share: fraction of purchases from seasons W16/S17 onward (recent vs classic)
import re

def parse_season_year(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r'(\d+)', str(s))
    if m:
        yr = int(m.group(1))
        return 2000 + yr if yr < 100 else yr
    return np.nan

transactions["season_year"] = transactions["prod_season"].apply(parse_season_year)
new_season_share = (
    transactions
    .assign(is_recent=(transactions["season_year"] >= 2016).astype(int))
    .groupby("cust_id")["is_recent"]
    .mean()
)
customer_features["new_season_share"] = customer_features["cust_id"].map(new_season_share).fillna(0)

# 9. Peak-quarter share: revenue in the customer's highest-spending quarter / total revenue
transactions["quarter"] = transactions["order_date"].dt.to_period("Q")
q_rev = (
    transactions[transactions["is_purchase"]]
    .groupby(["cust_id", "quarter"])["sale_revenue"]
    .sum()
    .reset_index()
)
q_max = q_rev.groupby("cust_id")["sale_revenue"].max()
customer_features["_peak_q_rev"] = customer_features["cust_id"].map(q_max).fillna(0)
customer_features["peak_quarter_share"] = (
    customer_features["_peak_q_rev"] /
    (customer_features["total_revenue"].clip(lower=1e-6))
).clip(0, 1)
customer_features = customer_features.drop(columns=["_peak_q_rev"])

print(f"Total features: {len(customer_features.columns) - 1}")  # -1 for cust_id


Total features: 45


In [52]:
# === Interaction & ratio features ===

# High-value + high-recency = the highest-risk customers
customer_features["recency_x_revenue"] = (
    customer_features["recency_days"] * customer_features["total_revenue"]
)

# Relative return damage: how much of gross value was returned
customer_features["return_damage_rate"] = (
    customer_features["return_revenue_lost"] /
    (customer_features["total_revenue"] + customer_features["return_revenue_lost"] + 1e-6)
)

# Spending trend relative to total revenue (normalised)
customer_features["spending_trend_relative"] = (
    customer_features["spending_trend"] /
    (customer_features["total_revenue"].abs() + 1e-6)
).clip(-3, 3)  # cap extreme outliers

# Customer loyalty signal: orders per active month
customer_features["orders_per_active_month"] = (
    customer_features["n_orders"] /
    (customer_features["active_months"] + 1)
)

# Revenue per active month (intensity of active periods)
customer_features["revenue_per_active_month"] = (
    customer_features["total_revenue"] /
    (customer_features["active_months"] + 1)
)

print(f"Total features: {len(customer_features.columns) - 1}")  # -1 for cust_id


Total features: 50


# New Features v2: Quarterly Breakdown, Recency Normalization, Target Encoding

In [53]:
# === Quarter-by-quarter breakdown ===
# 6 quarters in the observation window: Q1 2016 – Q2 2017
# For each: total revenue, unique orders, return count per customer
# Captures spending TRAJECTORY rather than just total aggregates

valid_quarters = ["2016Q1","2016Q2","2016Q3","2016Q4","2017Q1","2017Q2"]

transactions["quarter"] = transactions["order_date"].dt.to_period("Q").astype(str)
tx_q = transactions[transactions["quarter"].isin(valid_quarters)].copy()

# Revenue per quarter (purchases + returns)
q_revenue = (
    tx_q.groupby(["cust_id","quarter"])["sale_revenue"]
    .sum()
    .unstack(fill_value=0)
    .add_prefix("q_rev_")
    .reindex(columns=[f"q_rev_{q}" for q in valid_quarters], fill_value=0)
)

# Orders per quarter
q_orders = (
    tx_q.groupby(["cust_id","quarter"])["sale_id"]
    .nunique()
    .unstack(fill_value=0)
    .add_prefix("q_ord_")
    .reindex(columns=[f"q_ord_{q}" for q in valid_quarters], fill_value=0)
)

# Returns per quarter
q_returns = (
    tx_q[tx_q["is_return"]]
    .groupby(["cust_id","quarter"])["sale_id"]
    .count()
    .unstack(fill_value=0)
    .add_prefix("q_ret_")
    .reindex(columns=[f"q_ret_{q}" for q in valid_quarters], fill_value=0)
)

customer_features = (
    customer_features
    .merge(q_revenue.reset_index(),  on="cust_id", how="left")
    .merge(q_orders.reset_index(),   on="cust_id", how="left")
    .merge(q_returns.reset_index(),  on="cust_id", how="left")
)

# Customers with zero activity in a quarter get 0
for col in [c for c in customer_features.columns if c.startswith(("q_rev_","q_ord_","q_ret_"))]:
    customer_features[col] = customer_features[col].fillna(0)

print(f"Quarterly features added. Total features: {len(customer_features.columns)-1}")


Quarterly features added. Total features: 68


In [54]:
# === Recency normalized ===
# Absolute recency_days ignores purchase cadence
# A customer with 180-day recency and a 60-day cycle is 3x overdue (likely churned)
# A customer with 180-day recency and a 200-day cycle is still within their normal window
customer_features["recency_normalized"] = (
    customer_features["recency_days"] /
    (customer_features["avg_days_between_orders"] + 1)
)
p99 = customer_features["recency_normalized"].quantile(0.99)
customer_features["recency_normalized"] = customer_features["recency_normalized"].clip(upper=p99)


In [55]:
# === Primary brand and category per customer (most-purchased) ===
purchases_only = transactions[transactions["is_purchase"]].copy()

cust_brand = (
    purchases_only.groupby("cust_id")["prod_brand"]
    .agg(lambda x: x.value_counts().index[0] if len(x) > 0 else "unknown")
    .reset_index()
    .rename(columns={"prod_brand": "primary_brand"})
)

cust_cat = (
    purchases_only.groupby("cust_id")["prod_type_3"]
    .agg(lambda x: x.value_counts().index[0] if len(x) > 0 else "unknown")
    .reset_index()
    .rename(columns={"prod_type_3": "primary_category"})
)

customer_features = (
    customer_features
    .merge(cust_brand, on="cust_id", how="left")
    .merge(cust_cat,   on="cust_id", how="left")
)
customer_features["primary_brand"]    = customer_features["primary_brand"].fillna("unknown")
customer_features["primary_category"] = customer_features["primary_category"].fillna("unknown")


In [56]:
# === Brand and category return rates (transaction-level, no target leakage) ===
# These are properties of the PRODUCTS, not the target variable
brand_return_rate = (
    transactions.groupby("prod_brand")["is_return"].mean()
    .reset_index()
    .rename(columns={"prod_brand": "primary_brand", "is_return": "brand_return_rate"})
)
cat_return_rate = (
    transactions.groupby("prod_type_3")["is_return"].mean()
    .reset_index()
    .rename(columns={"prod_type_3": "primary_category", "is_return": "category_return_rate"})
)

global_return_rate = transactions["is_return"].mean()

customer_features = (
    customer_features
    .merge(brand_return_rate, on="primary_brand",    how="left")
    .merge(cat_return_rate,   on="primary_category", how="left")
)
customer_features["brand_return_rate"]    = customer_features["brand_return_rate"].fillna(global_return_rate)
customer_features["category_return_rate"] = customer_features["category_return_rate"].fillna(global_return_rate)


In [57]:
# === Target encoding with 5-fold CV (avoids label leakage) ===
# For each customer we encode their primary_brand / primary_category as
# the mean revenue_2018_2019 for that brand/category, computed on OTHER folds only.
# Test set: encoded with full training means + Laplace smoothing.

from sklearn.model_selection import KFold

train_df  = pd.read_csv("../data/customer_clv_train.csv")
test_df   = pd.read_csv("../data/customer_clv_test.csv")

train_feat = customer_features.merge(train_df, on="cust_id", how="inner").copy()
test_feat  = customer_features[customer_features["cust_id"].isin(test_df["cust_id"])].copy()

MIN_SAMPLES = 5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in ["primary_brand", "primary_category"]:
    enc_col = f"{col.split('_')[1]}_target_enc"  # brand_target_enc / category_target_enc
    global_mean = train_feat["revenue_2018_2019"].mean()

    oof_enc = pd.Series(index=train_feat.index, dtype=float)

    for _, (tr_idx, val_idx) in enumerate(kf.split(train_feat)):
        fold_tr  = train_feat.iloc[tr_idx]
        fold_val = train_feat.iloc[val_idx]

        cat_stats = fold_tr.groupby(col)["revenue_2018_2019"].agg(["mean","count"])

        def smooth_encode(cat):
            if cat not in cat_stats.index:
                return global_mean
            n   = cat_stats.loc[cat, "count"]
            mu  = cat_stats.loc[cat, "mean"]
            return (n * mu + MIN_SAMPLES * global_mean) / (n + MIN_SAMPLES)

        oof_enc.iloc[val_idx] = fold_val[col].apply(smooth_encode).values

    # Full training means for test encoding
    full_stats = train_feat.groupby(col)["revenue_2018_2019"].agg(["mean","count"])
    def smooth_full(cat):
        if cat not in full_stats.index:
            return global_mean
        n  = full_stats.loc[cat, "count"]
        mu = full_stats.loc[cat, "mean"]
        return (n * mu + MIN_SAMPLES * global_mean) / (n + MIN_SAMPLES)

    test_enc = test_feat[col].apply(smooth_full)

    # Write back to customer_features
    customer_features.loc[train_feat.index, enc_col] = oof_enc.values
    customer_features.loc[test_feat.index, enc_col]  = test_enc.values

customer_features["brand_target_enc"]    = customer_features["brand_target_enc"].fillna(train_df["revenue_2018_2019"].mean())
customer_features["category_target_enc"] = customer_features["category_target_enc"].fillna(train_df["revenue_2018_2019"].mean())

print(f"Target encoding done. Total features: {len(customer_features.columns)-1}")


Target encoding done. Total features: 75


In [58]:
# Drop string columns before saving (they've been encoded)
save_df = customer_features.drop(columns=["primary_brand","primary_category"], errors="ignore")

save_df.to_csv("../data/customer_features_v2.csv", index=False)
print(f"Saved customer_features_v2.csv — {len(save_df.columns)-1} features, {len(save_df)} customers")


Saved customer_features_v2.csv — 73 features, 145739 customers


In [59]:
customer_features.to_csv("../data/customer_features.csv", index=False)